# 04 - Memoria e stato

La memoria non è una proprietà magica del modello. È stato che decidiamo di passare di nuovo al modello.

In questo notebook vediamo tre livelli:

1. nessuna memoria: chiamate indipendenti
2. memoria manuale: una lista Python di messaggi
3. memoria LangGraph: checkpoint con `thread_id`

## Obiettivi
- capire perché le chiamate dirette sono stateless
- gestire una history manuale
- usare checkpoint LangGraph
- separare conversazioni con thread diversi


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "lab_agentic").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from lab_agentic.toolkit import LAB_TOOLS
from lab_agentic.utils import get_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START, StateGraph
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition


In [2]:
PROVIDER = "ollama"        # prova anche: "ollama_cloud"
MODEL_NAME = None

llm = get_chat_model(provider=PROVIDER, model=MODEL_NAME, temperature=0)


## Nessuna memoria

Queste sono due chiamate indipendenti. La seconda non sa nulla della prima, a meno che tu non glielo ripeta.


In [3]:
first = llm.invoke([
    SystemMessage(content="Sei un assistente utile e rispondi in italiano."),
    HumanMessage(content="Mi chiamo Giulia e voglio costruire agenti per aiutare la segreteria studenti."),
])
print(first.content)

second = llm.invoke([
    SystemMessage(content="Sei un assistente utile e rispondi in italiano."),
    HumanMessage(content="Come mi chiamo e che progetto voglio fare?"),
])
print()
print("SECONDA CHIAMATA:")
print(second.content)


Ciao Giulia! Sembra un progetto interessante!

Per costruire un agente di assistenza per la segreteria degli studenti, potresti considerare le seguenti fasi:

1. **Definisci gli obiettivi**: Qual è lo scopo dell'agente? Vuoi aiutare gli studenti a trovare informazioni, risolvere problemi o semplicemente fornire supporto generale?
2. **Identifica le funzionalità**: Quale tipo di assistenza vuoi offrire agli studenti? Ad esempio:
 * Rispondere alle domande più comuni (ad esempio, orari delle lezioni, informazioni sui corsi, ecc.)
 * Guidare gli studenti attraverso il processo di iscrizione o richiesta di documenti
 * Fornire informazioni su borse di studio, finanziamenti e altre risorse disponibili
 * Aiutare gli studenti a pianificare la loro carriera e scegliere i corsi più adatti alle loro esigenze
3. **Scegli una piattaforma**: Qual è il mezzo di comunicazione preferito degli studenti? Ad esempio:
 * Chatbot integrato nel sito web della segreteria
 * App mobile per gli studenti
 * Bo

## Memoria manuale

Una chat history è una lista di messaggi. I framework automatizzano questa idea, ma conviene vederla a mano una volta.


In [4]:
# In questo modo andiamo ad estendere la memoria del modello. Quello che succede è che in questo modo,
# ogni chiamata che facciamo al modello gli passiamo tutta la conversazione, per questo le performance degradano all'aumenttare della conversazione.
history = [SystemMessage(content="Sei un tutor universitario. Rispondi in italiano e sii breve.")]


def manual_chat(user_text: str):
    history.append(HumanMessage(content=user_text))
    response = llm.invoke(history)
    history.append(response)
    print(response.content)
    return response

manual_chat("Mi chiamo Giulia e mi interessa un progetto su chatbot per eventi del campus.")
manual_chat("Cosa ricordi del mio progetto?")


Ciao Giulia! Sembra un progetto interessante!

Per iniziare, potresti considerare le seguenti idee:

1. **Creazione di un chatbot per informazioni sugli eventi**: il bot potrebbe fornire informazioni sui prossimi eventi del campus, inclusa la data, l'ora e la descrizione dell'evento.
2. **Registrazione agli eventi**: il bot potrebbe permettere ai studenti di registrarsi agli eventi selezionati, inviando una conferma via email o SMS.
3. **Feedback degli studenti**: il bot potrebbe chiedere ai partecipanti a un evento di fornire feedback sull'evento stesso.

Qual è la tua idea principale per questo progetto?
Ricordo che il tuo progetto riguarda la creazione di un chatbot per eventi del campus universitario. Il tuo obiettivo è sviluppare un sistema che fornisca informazioni sugli eventi, permetta la registrazione agli eventi e raccolga feedback dagli studenti partecipanti.


AIMessage(content='Ricordo che il tuo progetto riguarda la creazione di un chatbot per eventi del campus universitario. Il tuo obiettivo è sviluppare un sistema che fornisca informazioni sugli eventi, permetta la registrazione agli eventi e raccolga feedback dagli studenti partecipanti.', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-05-14T10:23:30.924437Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6607407708, 'load_duration': 175203958, 'prompt_eval_count': 240, 'prompt_eval_duration': 309477541, 'eval_count': 73, 'eval_duration': 6064676871, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019e2603-3298-7aa2-83bb-a8fa9681ef3c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 240, 'output_tokens': 73, 'total_tokens': 313})

## Memoria LangGraph con thread id

Finora la memoria era esplicita: avevamo una lista Python `history` e la passavamo noi al modello a ogni chiamata.

Con LangGraph possiamo spostare questa responsabilita nel grafo usando un **checkpointer**. Il checkpointer salva lo stato del grafo dopo ogni passo. In questo esempio lo stato e `MessagesState`, quindi contiene la lista dei messaggi della conversazione.

Il punto importante e `thread_id`: non e un id tecnico qualunque, ma la chiave della conversazione. Quando chiamiamo il grafo con lo stesso `thread_id`, LangGraph recupera lo stato precedente, aggiunge il nuovo messaggio e continua. Quando cambiamo `thread_id`, parte o continua un'altra conversazione separata.

In breve:

1. `MessagesState` definisce cosa viene salvato: i messaggi
2. `InMemorySaver()` decide dove salvarlo: in memoria Python, solo per questa sessione
3. `thread_id` decide quale conversazione leggere e aggiornare
4. `invoke(...)` riceve solo il nuovo messaggio, ma il grafo ricostruisce il contesto dal checkpoint


In [5]:
CHAT_SYSTEM = SystemMessage(
    content=(
        "Sei un assistente universitario. Rispondi in italiano e in modo breve. "
        "Usa la cronologia quando l'utente si riferisce a messaggi precedenti."
    )
)


# Il nodo riceve lo stato corrente del grafo.
# Con MessagesState, state["messages"] contiene tutta la conversazione
# recuperata dal checkpointer piu il nuovo messaggio dell'utente.
def memory_assistant(state: MessagesState):
    response = llm.invoke([CHAT_SYSTEM] + state["messages"])

    # Restituiamo solo il nuovo messaggio dell'assistente.
    # LangGraph lo aggiunge alla lista esistente di messaggi nello stato.
    return {"messages": [response]}


builder = StateGraph(MessagesState)
builder.add_node("assistant", memory_assistant)
builder.add_edge(START, "assistant")

# InMemorySaver salva i checkpoint in RAM.
# Va bene per capire il meccanismo in aula, ma non sopravvive al riavvio
# del kernel. In produzione useremmo un checkpointer persistente.
checkpointer = InMemorySaver()
memory_graph = builder.compile(checkpointer=checkpointer)


In [6]:
def chat(question: str, thread_id: str):
    # Il config non e parte del prompt: e metadato per LangGraph.
    # Serve al checkpointer per sapere quale stato caricare e aggiornare.
    config = {"configurable": {"thread_id": thread_id}}

    # Passiamo solo il nuovo HumanMessage.
    # Se esiste gia un checkpoint per questo thread_id, LangGraph lo recupera
    # e combina i vecchi messaggi con questo nuovo input.
    result = memory_graph.invoke(
        {"messages": [HumanMessage(content=question)]},
        config=config,
    )

    print(result["messages"][-1].content)
    return result


# Prima chiamata: crea lo stato per il thread studentessa-giulia.
chat("Mi chiamo Giulia. Voglio fare un agente che consiglia eventi del campus.", thread_id="studentessa-giulia")
print("--------------------")
# Seconda chiamata con lo stesso thread_id: il grafo ritrova il nome e il progetto.
chat("Che progetto voglio fare?", thread_id="studentessa-giulia")


Ciao Giulia! Sembra un progetto interessante.

Per iniziare, potresti dirmi cosa intendi per "agenzia di consulenza" e quali tipi di eventi del campus vorresti promuovere?
--------------------
Il tuo progetto! Vorresti creare un agente virtuale che consiglia agli studenti degli eventi del campus, come concerti, spettacoli teatrali, mostre d'arte o attività sportive. L'agente dovrebbe essere in grado di rispondere alle domande degli studenti e suggerire loro gli eventi più adatti alle loro preferenze.


{'messages': [HumanMessage(content='Mi chiamo Giulia. Voglio fare un agente che consiglia eventi del campus.', additional_kwargs={}, response_metadata={}, id='7b10a8c1-d6dc-4bc6-a395-2123cafa2073'),
  AIMessage(content='Ciao Giulia! Sembra un progetto interessante.\n\nPer iniziare, potresti dirmi cosa intendi per "agenzia di consulenza" e quali tipi di eventi del campus vorresti promuovere?', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-05-14T10:23:46.859784Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5314864625, 'load_duration': 111199750, 'prompt_eval_count': 73, 'prompt_eval_duration': 975505833, 'eval_count': 53, 'eval_duration': 4192146995, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019e2603-75e6-7cd0-a70f-ed25fe5c722d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 73, 'output_tokens': 53, 'total_tokens': 126}),
  HumanMessage(content='Che progetto voglio f

## Conversazioni separate

Ora usiamo lo stesso identico grafo, ma cambiamo `thread_id`.

Questo mostra il comportamento piu importante del checkpointer: la memoria non e globale. E separata per conversazione. Se Marco usa un thread diverso, non eredita le informazioni di Giulia; se torniamo al thread di Giulia, ritroviamo il suo stato.


In [ ]:
# Marco non ha ancora detto niente in questo thread.
# Il modello non dovrebbe conoscere il progetto di Giulia.
chat("Che progetto voglio fare?", thread_id="studente-marco")

# Ora creiamo memoria anche per Marco.
chat("Mi chiamo Marco e voglio lavorare su test automatici per agenti.", thread_id="studente-marco")
chat("Che progetto voglio fare?", thread_id="studente-marco")

# Tornando al thread di Giulia, il grafo ricarica la conversazione di Giulia.
chat("Che progetto voglio fare?", thread_id="studentessa-giulia")

Non hai ancora menzionato alcun progetto specifico. Se vuoi, possiamo discuterne insieme per trovare un'idea che ti piaccia!
Interessante! Hai già iniziato a lavorarci? Qual è il tuo obiettivo principale con questo progetto? (Ricorda che posso rispondere in base ai messaggi precedenti se ne hai bisogno)
Ricordo! Hai detto che vuoi lavorare su test automatici per agenti. Quindi, il tuo progetto è proprio questo: sviluppare un sistema di test automatizzati per verificare la correttezza e l'efficienza degli agenti in un ambiente di intelligenza artificiale.
Ricordo! Il tuo progetto è un agente virtuale che consiglia agli studenti degli eventi del campus, come concerti, spettacoli teatrali, mostre d'arte o attività sportive. Vuoi iniziare a pianificare le funzionalità dell'agente?


{'messages': [HumanMessage(content='Mi chiamo Giulia. Voglio fare un agente che consiglia eventi del campus.', additional_kwargs={}, response_metadata={}, id='7b10a8c1-d6dc-4bc6-a395-2123cafa2073'),
  AIMessage(content='Ciao Giulia! Sembra un progetto interessante.\n\nPer iniziare, potresti dirmi cosa intendi per "agenzia di consulenza" e quali tipi di eventi del campus vorresti promuovere?', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-05-14T10:23:46.859784Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5314864625, 'load_duration': 111199750, 'prompt_eval_count': 73, 'prompt_eval_duration': 975505833, 'eval_count': 53, 'eval_duration': 4192146995, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019e2603-75e6-7cd0-a70f-ed25fe5c722d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 73, 'output_tokens': 53, 'total_tokens': 126}),
  HumanMessage(content='Che progetto voglio f

## Ispeziona lo stato salvato

Possiamo guardare dentro il checkpoint per rendere la memoria concreta. `get_state(config)` recupera l'ultimo stato associato a quel `thread_id`.

Qui stampiamo gli ultimi messaggi salvati per Giulia. Dovresti vedere alternanza di `HumanMessage` e `AIMessage`: sono proprio i messaggi che il grafo rimettera nel contesto alla prossima chiamata con lo stesso thread.


In [8]:
config = {"configurable": {"thread_id": "studentessa-giulia"}}
state = memory_graph.get_state(config)

print("Chiavi nello stato:", state.values.keys())
print("Messaggi salvati:", len(state.values["messages"]))
print()

for message in state.values["messages"][-6:]:
    print(type(message).__name__, "-", message.content[:200])


Chiavi nello stato: dict_keys(['messages'])
Messaggi salvati: 6

HumanMessage - Mi chiamo Giulia. Voglio fare un agente che consiglia eventi del campus.
AIMessage - Ciao Giulia! Sembra un progetto interessante.

Per iniziare, potresti dirmi cosa intendi per "agenzia di consulenza" e quali tipi di eventi del campus vorresti promuovere?
HumanMessage - Che progetto voglio fare?
AIMessage - Il tuo progetto! Vorresti creare un agente virtuale che consiglia agli studenti degli eventi del campus, come concerti, spettacoli teatrali, mostre d'arte o attività sportive. L'agente dovrebbe essere
HumanMessage - Che progetto voglio fare?
AIMessage - Ricordo! Il tuo progetto è un agente virtuale che consiglia agli studenti degli eventi del campus, come concerti, spettacoli teatrali, mostre d'arte o attività sportive. Vuoi iniziare a pianificare le


## Tool agent con memoria

Il checkpointer non e legato al caso semplice senza tool. Possiamo usarlo anche con un grafo agentico: nodo assistente, nodo tool e arco condizionale.

La differenza e che lo stato ora puo contenere anche messaggi tecnici prodotti dal tool loop, per esempio richieste di tool call e `ToolMessage`. Questo e utile per riferimenti come "il primo corso" o "ripeti la risposta precedente", ma rende ancora piu importante ispezionare lo stato quando qualcosa non torna.


In [9]:
llm_with_tools = llm.bind_tools(LAB_TOOLS)

TOOL_MEMORY_SYSTEM = SystemMessage(
    content=(
        "Sei un assistente per orientamento corsi. Rispondi in italiano. "
        "Usa tool per fatti su corsi, aule, policy, orari ed eventi. "
        "Usa la cronologia per capire riferimenti come 'quel corso' o 'il primo'."
    )
)


def tool_memory_assistant(state: MessagesState):
    # Anche qui state["messages"] arriva dal checkpoint del thread corrente.
    # Se in una chiamata precedente il grafo ha usato tool, nello stato possono
    # esserci anche AIMessage con tool_calls e ToolMessage con i risultati.
    response = llm_with_tools.invoke([TOOL_MEMORY_SYSTEM] + state["messages"])
    return {"messages": [response]}


builder = StateGraph(MessagesState)
builder.add_node("assistant", tool_memory_assistant)
builder.add_node("tools", ToolNode(LAB_TOOLS))
builder.add_edge(START, "assistant")

# Se l'assistente chiede un tool, si va al nodo tools.
# Altrimenti il grafo termina e il checkpoint salva la risposta finale.
builder.add_conditional_edges("assistant", tools_condition)

# Dopo il tool, si torna all'assistente per trasformare il risultato grezzo
# in una risposta naturale per l'utente.
builder.add_edge("tools", "assistant")

tool_memory_graph = builder.compile(checkpointer=InMemorySaver())


In [10]:
def advisor(question: str, thread_id: str = "advisor-giulia"):
    config = {"configurable": {"thread_id": thread_id}}
    result = tool_memory_graph.invoke(
        {"messages": [HumanMessage(content=question)]},
        config=config,
    )
    print(result["messages"][-1].content)
    return result


# La prima domanda introduce due corsi nel thread advisor-giulia.
advisor("Sto scegliendo tra AI301 e SE220. Si sovrappongono?", thread_id="advisor-giulia")

# Qui "il primo corso" funziona solo se la cronologia precedente viene recuperata.
advisor("Quale aula usa il primo corso e ha registrazione?", thread_id="advisor-giulia")

# Qui chiediamo di riusare memoria conversazionale, non solo dati dai tool.
advisor("Ripetimi la risposta sulla sovrapposizione.", thread_id="advisor-giulia")


Sì, i due corsi sovrappongono una sessione di lezione mercoledì dalle 10:00 alle 11:00. Potresti controllare gli orari aggiornati per essere sicuro che non ci siano altre sovrapposizioni.
{"name": "room_recommendation_tool", "parameters": {"min_capacity": 0, "needs_recording": True}}
Sì, i due corsi sovrappongono una sessione di lezione mercoledì dalle 10:00 alle 11:00. Potresti controllare gli orari aggiornati per essere sicuro che non ci siano altre sovrapposizioni.


{'messages': [HumanMessage(content='Sto scegliendo tra AI301 e SE220. Si sovrappongono?', additional_kwargs={}, response_metadata={}, id='09b2a794-c0ca-4346-8b14-f2e64fa74dd2'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-05-14T10:25:46.97279Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7413872000, 'load_duration': 138472416, 'prompt_eval_count': 571, 'prompt_eval_duration': 5024937459, 'eval_count': 29, 'eval_duration': 2221532791, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019e2605-42e2-78f2-8336-619a333dd3e2-0', tool_calls=[{'name': 'schedule_conflict_tool', 'args': {'code_a': 'AI301', 'code_b': 'SE220'}, 'id': 'a5a48cad-9427-4603-944a-15ce49f26610', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 571, 'output_tokens': 29, 'total_tokens': 600}),
  ToolMessage(content="{'conflict': True, 'overlap_slots': ['Wed 10:00-11:00'], 'reason'

## Sfida

Prova questi esperimenti:

- cambia solo `thread_id` e osserva cosa sparisce
- usa riferimenti come `quel corso`, `il primo`, `la seconda aula`
- ispeziona lo stato dopo una tool call
- riavvia il kernel e verifica cosa succede con `InMemorySaver`
- aggiungi un messaggio di riassunto e valuta se aiuta i modelli locali piccoli

Domanda guida: il modello sta ricordando davvero, oppure stiamo reinserendo nel prompt uno stato salvato dal grafo?
